In [ ]:
import pandas as pd
import numpy as np
from tools import sherlock

In [ ]:
df = pd.read_csv(r"..\data\raw\2BLONDEL_2024.csv", sep=';', encoding='latin_1', decimal=',')
df.columns = df.columns.str.strip().str.replace(' ', '_')
df['Date_écriture'] = pd.to_datetime(df['Date_écriture'], format="%d/%m/%Y", dayfirst=True)

# Exploration

In [ ]:
sherlock(df)

In [ ]:
df = df.drop(columns=['Débit_origine', 'Crédit_origine', 'Lettrage_partiel', 'Monnaie', 'ISO_Monnaie', 'taux_change'])

In [ ]:
df.head()

In [ ]:
df['Lettrage_n+1'].value_counts()

In [ ]:
# Les codes compte uniques avec leur intitulé
df.groupby(['Code_compte', 'Intitulé_compte']).size().sort_values(ascending=False).reset_index()

In [ ]:
# Répartition par journal
df.groupby('Code_journal').size()

In [ ]:
df.query("Code_journal == 'AC'")

In [ ]:
# Quelques lignes par journal pour voir les patterns
df.query("Code_journal == 'BQ'").head(10)

# CA

In [ ]:
df.query("Code_compte == '70600000'")['Crédit_euro'].sum()

## CA Mensuel

In [ ]:
# CA mensuel
df.query("Code_compte == '70600000'").groupby(df['Date_écriture'].dt.to_period('M'))['Crédit_euro'].sum()


## CA par PTF

In [ ]:
loyers = df.query("Code_compte == '70600000'")

In [ ]:
loyers['Libellé_écriture'].unique()

In [ ]:
conds = [
    loyers['Libellé_écriture'].str.contains('BNB', case=False),
    loyers['Libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

In [ ]:
# CA par PTF

loyers.groupby('canal')['Crédit_euro'].sum()

In [ ]:
# CA par PTF par mois

loyers.groupby([loyers['Date_écriture'].dt.to_period('M'), 'canal'])['Crédit_euro'].sum()

# Dépenses

In [ ]:
df['Débit_euro'].sum()

## Consommables

In [ ]:
conso = df.query("Code_compte == '60630000'")
conso.groupby(['Code_compte', 'Intitulé_compte'])['Débit_euro'].sum()

## Conso mois

In [ ]:
conso.groupby([conso['Date_écriture'].dt.to_period('M'), 'Code_compte', 'Intitulé_compte'])['Débit_euro'].sum()

## Commission

In [ ]:
df.query("Code_compte == '62220000'")

## Dépense par categ

In [ ]:
df.groupby(['Code_compte', 'Intitulé_compte'])['Débit_euro'].sum()

## Depense par personnes

In [ ]:
conds = [
    df['Libellé_écriture'].str.contains('BLONDEL', case=False),
    df['Libellé_écriture'].str.contains('MOIZAN', case=False),
    ]
choices = ['BLONDEL', 'MOIZAN']
df['qui'] = np.select(conds, choices, default='AUTRE')

df.groupby('qui')['Débit_euro'].sum()

In [ ]:
df.query("Code_compte == 'FACTION'")

In [ ]:
df[df['Code_compte'].str.startswith('F')].head(50)